# 00. AI Hub 데이터 받기 (Colab)

<a href="https://colab.research.google.com/github/2026-2nd/26-2-1/blob/main/notebooks/00_aihub_download.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

코드는 `src/aihub_download.py`, 받을 파일 목록은 `configs/aihub_71385.json`에 있다. 이용 조건은 `docs/DATA_SOURCES.md`를 따른다.

**준비**: AI Hub에서 다운로드 신청 승인 → Colab 🔑 Secrets에 `AIHUB_APIKEY` 등록 (노트북 액세스 허용). 키를 셀에 직접 쓰지 않는다.

**저장 위치**: zip 원본은 Drive `MyDrive/ML_TeamWork/data/raw/aihub_71385/`에 보관하고 `MANIFEST.csv`에 크기·SHA-256을 기록한다. 학습할 때는 `/content/data`로 압축을 푼다.

In [ ]:
from google.colab import drive, userdata
import os, subprocess, sys
drive.mount('/content/drive')

REPO_DIR = '/content/drive/MyDrive/ML_TeamWork/26-2-1'   # Drive에 복제한 저장소 (Drive에서 코드가 보임)
if os.path.isdir(os.path.join(REPO_DIR, '.git')):
    subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/2026-2nd/26-2-1.git', REPO_DIR], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

os.environ['ML_TEAMWORK_DATA'] = '/content/drive/MyDrive/ML_TeamWork/data'
os.environ['AIHUB_APIKEY'] = userdata.get('AIHUB_APIKEY')
!df -h /content | tail -1

In [ ]:
# 받을 수 있는 묶음(preset)과 AI Hub 파일 목록(파일명 | 용량 | filekey)
!python -m src.aihub_download presets
!python -m src.aihub_download list | tail -n +10 | head -40

In [ ]:
# 실내형분류기 전체(학습 ~17GB + 검증 ~2GB). 이미 받은 파일은 MANIFEST.csv를 보고 건너뛴다. 동작만 확인하려면 --preset smoke_test
# 예: --preset labels_all --preset indoor_val_img   /   --filekey 514423,514447
# 임시 파일은 Colab 로컬 디스크(/content/aihub_tmp)에 두고, 완성된 zip만 Drive로 옮긴다.
!python -m src.aihub_download download --preset indoor_train_img --preset indoor_val_img --work-dir /content/aihub_tmp

## 학습 세션마다: Drive의 zip → `/content/data`로 압축 해제

In [ ]:
!python -m src.aihub_download extract --pattern '*실내형분류기*' --dest /content/data/aihub_71385
os.environ['ML_TEAMWORK_DATA'] = '/content/data'  # 이후 학습 코드는 로컬 디스크에서 읽음